In [2]:
!pip install -q transformers accelerate datasets torch codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.8/380.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.2 MB/s eta 0:00:00


In [3]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 45.4 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Model

In [7]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [8]:
import torch
import os
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/gemma-3n-E4B-it"
os.makedirs("/content/offload", exist_ok=True)

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    offload_folder="/content/offload"
)
model.eval()
print("Ready.")

processor_config.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1676 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Ready.


## Load VQAv2 Sample

In [9]:
from datasets import load_dataset
import random

ds = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True)

samples = []
for item in ds:
    samples.append(item)
    if len(samples) >= 200:
        break

yes_no = [s for s in samples if s["answer_type"] == "yes/no"][:4]
number = [s for s in samples if s["answer_type"] == "number"][:3]
other  = [s for s in samples if s["answer_type"] == "other"][:3]
stratified = yes_no + number + other
random.shuffle(stratified)
print(f"Sample size: {len(stratified)}")

README.md:   0%|          | 0.00/962 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Sample size: 10


for 50 samples:

In [13]:
from datasets import load_dataset
import random

ds = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True)

samples = []
for item in ds:
    samples.append(item)
    if len(samples) >= 500:
        break

yes_no = [s for s in samples if s["answer_type"] == "yes/no"][:15]
number = [s for s in samples if s["answer_type"] == "number"][:15]
other  = [s for s in samples if s["answer_type"] == "other"][:20]
stratified = yes_no + number + other
random.shuffle(stratified)
print(f"Sample size: {len(stratified)}")

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Sample size: 50


## Inference + Logging

Gemma 4B

In [14]:
import time
from codecarbon import EmissionsTracker
import torch

tracker = EmissionsTracker(log_level="error")
tracker.start()

results = []

for i, item in enumerate(stratified):
    image = item["image"]
    question = item["question"]

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"Question: {question} Answer briefly."}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )
    ttft = time.perf_counter() - start

    predicted = processor.decode(output[0], skip_special_tokens=True).strip()
    gt = item["multiple_choice_answer"]

    results.append({
        "question": question,
        "predicted": predicted,
        "ground_truth": gt,
        "answer_type": item["answer_type"],
        "ttft_seconds": round(ttft, 4)
    })

    print(f"[{i+1}/50] Q: {question[:40]} | GT: {gt} | Pred: {predicted[-40:]}")

emissions = tracker.stop()
print(f"\nDone. Emissions: {emissions:.6f} kg CO2")

[1/50] Q: What color are the walls? | GT: white | Pred:  the walls? Answer briefly.
model
Beige.
[2/50] Q: How many beds? | GT: 1 | Pred: ow many beds? Answer briefly.
model
One.
[3/50] Q: What kind of room is this? | GT: bedroom | Pred:  is this? Answer briefly.
model
Bedroom.
[4/50] Q: How many are playing ball? | GT: 1 | Pred: playing ball? Answer briefly.
model
One.
[5/50] Q: Is this rice noodle soup? | GT: yes | Pred:  noodle soup? Answer briefly.
model
Yes.
[6/50] Q: How many seats are there? | GT: 1 | Pred: ts are there? Answer briefly.
model
One.
[7/50] Q: What is the child eating? | GT: donut | Pred: d eating? Answer briefly.
model
A donut.
[8/50] Q: How many umbrellas do you see? | GT: 0 | Pred:  **two** umbrellas visible in the image.
[9/50] Q: How many people are on the field? | GT: 3 | Pred: There are **three** people on the field.
[10/50] Q: How many frames are on the wall? | GT: 7 | Pred: odel
There are **6** frames on the wall.
[11/50] Q: What color is the kids hair? | 

Baseline Results

In [15]:
import json

correct = sum(
    1 for r in results
    if r["ground_truth"].lower() in r["predicted"].lower()
)
vqa_score = correct / len(results)
avg_ttft  = sum(r["ttft_seconds"] for r in results) / len(results)

baseline = {
    "model": "gemma-3n-E4B",
    "n_samples": len(results),
    "vqa_accuracy": round(vqa_score, 4),
    "avg_ttft_seconds": round(avg_ttft, 4),
    "total_kg_co2": round(emissions, 8),
    "kg_co2_per_sample": round(emissions / len(results), 10),
    "results": results
}

with open("baseline_results.json", "w") as f:
    json.dump(baseline, f, indent=2)

print(f"VQA Accuracy : {vqa_score:.2%}")
print(f"Avg TTFT     : {avg_ttft:.3f}s")
print(f"CO2 total    : {emissions:.6f} kg")
print(f"CO2/sample   : {emissions/len(results):.8f} kg")

VQA Accuracy : 36.00%
Avg TTFT     : 12.440s
CO2 total    : 0.006200 kg
CO2/sample   : 0.00012401 kg


In [17]:
from google.colab import files
files.download("baseline_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>